In [2]:
# 관련 모듈과 라이브러리 import하기

# nn: 신경망 만들 때 쓰는 기본 도구들 (레이어, 활성화 함수 등)
import torch.nn as nn

# torch: 텐서 다루고 GPU 쓸 수 있게 해준다
import torch 

# datasets: 유명한 데이터셋 쉽게 다운받게 해줌
from torchvision import datasets

# transforms: 이미지를 모델이 먹을 수 있게 변환해줌(텐서로 변환하고 정규화)
from torchvision import transforms

# DataLoader: 데이터를 불러오는 애
from torch.utils.data import DataLoader

# numpy: 숫자 계산할 때 쓰는 라이브러리
import numpy as np

# optim: 모델을 학습시키는 최적화 알고리즘들 (경사하강법 등)
import torch.optim as optim 

# plt: 그래프 그리고 이미지 보여주는 애
import matplotlib.pyplot as plt

# 모델 불러오기
from models import CNN

# 공격 기법 불러오기
from attacks import fgsm_attack, ifgsm_attack

In [3]:
# gpu 사용 위해 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [4]:
# 불러올 모델과 같은 구조를 정의하기
model = CNN()
# 저장해두었던 가중치, bias 불러오기 
model.load_state_dict(torch.load('cifar10_cnn_model.pth'))
# gpu로 모델 옮기기 
model.to(device)
# 평가 모드로 전환
# 레이어 동작이 학습할 때랑 다르게 바뀜 (예: Dropout, BatchNorm)
# Dropout: 학습할 때는 일부 뉴런 끄지만, 평가할 때는 모두 사용
# BatchNorm: 학습할 때는 미니배치 통계 사용하지만, 평가할 때는 전체 데이터셋 통계 사용한다!
model.eval()

/tmp/ipykernel_288872/4197865864.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('cifar10_cnn_model.pth'))


CNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=2048, out_features=10, bias=True)
  (relu): ReLU()
)

In [5]:
# 테스트 데이터셋 불러오자.

# 데이터셋 저장 경로
download_root = 'CIFAR10_data/'

#정규화 
transformations = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])
# 테스트용 데이터셋
dataset2 = datasets.CIFAR10(root=download_root,
                            # 학습용이 아니므로 이번에는 False!
                            train = False, 
                            transform = transformations,
                            download=False)

In [8]:
# 공격 대상 데이터로더 만들기
test_loader = DataLoader(dataset2,
                        batch_size=1,
                        shuffle=False)

# 클래스 이름들
classes = ('plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')

In [12]:
# 이미지 한 장 가져오기
for image, label in test_loader:
    # 배치에서 이미지와 레이블 가져오기
    image, label = image.to(device), label.to(device)
    break # 첫 번째 배치만 사용하기 위해 루프 종료 -> 이미지 1장과 레이블 1개만 있음

# 원본 이미지로 모델 예측해보기
outputs = model(image)
_, predicted = torch.max(outputs.data, 1)
print(f'원본 이미지 예측: {classes[predicted.item()]}, 실제 레이블: {classes[label.item()]}')

# FGSM 공격 적용
epsilon = 0.1  # 공격 강도 설정

# 해당 이미지에 대한 gradeint 계산

#이미지에 대한 gradient 계산을 위해 requires_grad 설정
image.requires_grad = True
# 모델 출력 계산
outputs = model(image)
loss_function = nn.CrossEntropyLoss()
loss = loss_function(outputs, label)
#∂Loss/∂image 계산
loss.backward()
data_grad = image.grad.data

# FGSM 공격 수행
adv_images = fgsm_attack(image, epsilon, data_grad)

# 공격된 이미지로 모델 예측해보기
outputs_adv = model(adv_images)
_, predicted_adv = torch.max(outputs_adv.data, 1)
print(f'공격된 이미지 예측: {classes[predicted_adv.item()]}, 실제 레이블: {classes[label.item()]}')

원본 이미지 예측: cat, 실제 레이블: cat
공격된 이미지 예측: ship, 실제 레이블: cat
